# Sesion 2 - Metro Madrid: estacionalidad compleja y prediccion practica

Continuacion conceptual de la sesion 1:
- En FX el foco era intervalo/riesgo en serie no estacionaria.
- Aqui trabajamos una serie con patrones estacionales marcados (semanal y anual), separando componente determinista y ruido.


## 1) Librerias

Usamos herramientas simples y utiles para analisis de series temporales en contexto real.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)



## 2) Carga y limpieza minima

Objetivo: tener serie diaria limpia y guardar version procesada.


In [ ]:
# Cargamos fichero original.
df = pd.read_csv("../data/raw/demanda_diaria_metro_2015_2024.csv")

# Parseamos fecha.
df["Fecha"] = pd.to_datetime(df["Fecha"])

# Renombramos para trabajar con nombres consistentes.
df = df.rename(columns={"Fecha": "date", "Demanda": "y"})

# Convertimos y a numerico y limpiamos nulos.
df["y"] = pd.to_numeric(df["y"], errors="coerce")
df = df.dropna(subset=["y"]).sort_values("date")

# Indice temporal diario.
df = df.set_index("date")

# Reindexamos a diario por robustez y rellenamos posibles huecos.
idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
df = df.reindex(idx)
df["y"] = df["y"].interpolate(method="time").ffill().bfill()

print("Filas:", len(df))
print("Inicio:", df.index.min())
print("Fin:", df.index.max())
print("Nulos:", df["y"].isna().sum())



In [ ]:
# Guardamos limpio para trazabilidad.
df.reset_index().rename(columns={"index": "date"}).to_csv("../data/processed/metro_daily_clean.csv", index=False)



## 3) EDA de dominio

Miramos serie completa, impacto COVID y patrones de calendario.


In [ ]:
# Marcamos periodo COVID para analisis visual.
covid_start = pd.Timestamp("2020-03-01")
covid_end = pd.Timestamp("2022-12-31")
df["is_covid"] = (df.index >= covid_start) & (df.index <= covid_end)

# Serie completa con banda COVID.
plt.figure(figsize=(14, 4))
plt.plot(df.index, df["y"], color="tab:blue", linewidth=1)
plt.axvspan(covid_start, covid_end, color="red", alpha=0.12, label="Periodo COVID")
plt.title("Demanda diaria Metro Madrid")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.legend()
plt.show()

# Variables de calendario para analisis estacional.
df_eda = df.copy()
df_eda["dow"] = df_eda.index.day_name()
df_eda["month"] = df_eda.index.month

# Patron semanal.
order_dow = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plt.figure(figsize=(10, 4))
sns.boxplot(data=df_eda, x="dow", y="y", order=order_dow)
plt.title("Distribucion por dia de semana")
plt.xlabel("Dia")
plt.ylabel("Viajeros")
plt.xticks(rotation=25)
plt.show()

# Patron anual por mes.
plt.figure(figsize=(10, 4))
sns.boxplot(data=df_eda, x="month", y="y")
plt.title("Distribucion por mes")
plt.xlabel("Mes")
plt.ylabel("Viajeros")
plt.show()



## 4) Descomposicion: tendencia + estacionalidad + ruido

Objetivo tecnico:
1. Separar componente determinista (tendencia + estacionalidad).
2. Aislar residuo (ruido/incertidumbre).
3. Cuantificar la incertidumbre para intervalos de confianza.


In [ ]:
# STL semanal para separar tendencia/estacionalidad/residuo.
stl = STL(df["y"], period=7, robust=True).fit()

# Componentes STL.
df["trend_stl"] = stl.trend
df["seasonal_weekly_stl"] = stl.seasonal
df["resid_stl"] = stl.resid

# Estacionalidad anual explicita: media por mes (factor anual aproximado).
month_factor = df.groupby(df.index.month)["y"].mean()
df["seasonal_annual_month"] = df.index.month.map(month_factor)

# Componente determinista aproximada = tendencia STL + estacionalidad semanal STL.
df["deterministic_part"] = df["trend_stl"] + df["seasonal_weekly_stl"]

# Ruido = observado - parte determinista.
df["noise_part"] = df["y"] - df["deterministic_part"]

# Graficos de componentes.
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(df.index, df["y"], color="black")
axes[0].set_title("Serie observada")

axes[1].plot(df.index, df["trend_stl"], color="tab:blue")
axes[1].set_title("Tendencia")

axes[2].plot(df.index, df["seasonal_weekly_stl"], color="tab:green")
axes[2].set_title("Estacionalidad semanal (STL)")

axes[3].plot(df.index, df["noise_part"], color="tab:red")
axes[3].set_title("Ruido / incertidumbre (residuo)")
plt.tight_layout()
plt.show()



In [ ]:
# Estadisticas del ruido para interpretar incertidumbre.
noise_std = df["noise_part"].std()
noise_mean = df["noise_part"].mean()

print(f"Media del ruido: {noise_mean:.2f}")
print(f"Desviacion tipica del ruido: {noise_std:.2f}")
print("Interpretacion: esta dispersion es la base para construir intervalos razonables.")



## 5) Forecast medio plazo: viajeros medios de los proximos 12 meses

Practical setup:
- Evaluamos primero en pseudo-test (ultimos 12 meses historicos).
- Comparamos modelos y calidad de intervalos.
- Luego proyectamos 12 meses futuros con el mejor enfoque.


In [ ]:
# Serie mensual (promedio de viajeros diarios por mes).
monthly = df["y"].resample("MS").mean()

# Pseudo-test: ultimos 12 meses.
train_m = monthly.iloc[:-12]
test_m = monthly.iloc[-12:]

# Modelo 1: SARIMA mensual con estacionalidad anual (12).
sarima_m = SARIMAX(train_m, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
sarima_m_fit = sarima_m.fit(disp=False)
fc_m = sarima_m_fit.get_forecast(steps=12)
sarima_m_pred = fc_m.predicted_mean
sarima_m_ci = fc_m.conf_int(alpha=0.05)
sarima_m_lo = sarima_m_ci.iloc[:, 0]
sarima_m_hi = sarima_m_ci.iloc[:, 1]

# Modelo 2 (bonus clasico especializado): Holt-Winters (ETS aditivo).
ets_m = ExponentialSmoothing(train_m, trend="add", seasonal="add", seasonal_periods=12).fit()
ets_m_pred = ets_m.forecast(12)

# Intervalo ETS aproximado por residuo (simple y practico).
ets_sigma = pd.Series(ets_m.resid).std()
ets_m_lo = ets_m_pred - 1.96 * ets_sigma
ets_m_hi = ets_m_pred + 1.96 * ets_sigma

# Metricas y cobertura.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

summary_m = pd.DataFrame(
    {
        "Modelo": ["SARIMA mensual", "ETS mensual"],
        "MAE": [
            mean_absolute_error(test_m, sarima_m_pred),
            mean_absolute_error(test_m, ets_m_pred),
        ],
        "RMSE": [
            rmse(test_m, sarima_m_pred),
            rmse(test_m, ets_m_pred),
        ],
        "Cobertura_IC95": [
            ((test_m >= sarima_m_lo) & (test_m <= sarima_m_hi)).mean(),
            ((test_m >= ets_m_lo) & (test_m <= ets_m_hi)).mean(),
        ],
    }
)
summary_m["%Fuera_IC95"] = 1 - summary_m["Cobertura_IC95"]
summary_m



In [ ]:
# Visual comparativa en pseudo-test mensual.
plt.figure(figsize=(14, 4))
plt.plot(test_m.index, test_m.values, label="Real", color="black")

plt.plot(test_m.index, sarima_m_pred.values, label="SARIMA", color="tab:green")
plt.fill_between(test_m.index, sarima_m_lo.values, sarima_m_hi.values, color="tab:green", alpha=0.2, label="IC95 SARIMA")

plt.plot(test_m.index, ets_m_pred.values, label="ETS", color="tab:orange")
plt.fill_between(test_m.index, ets_m_lo.values, ets_m_hi.values, color="tab:orange", alpha=0.15, label="IC95 ETS")

plt.title("Validacion mensual (ultimos 12 meses)")
plt.xlabel("Fecha")
plt.ylabel("Viajeros medios diarios del mes")
plt.legend(loc="best")
plt.show()



In [ ]:
# Forecast real: proximos 12 meses usando todo el historico.
monthly_full = monthly.copy()

# Reentrenamos SARIMA con toda la serie.
sarima_full = SARIMAX(monthly_full, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
fc_12 = sarima_full.get_forecast(steps=12)
future_m = fc_12.predicted_mean
future_m_ci = fc_12.conf_int(alpha=0.05)

future_m_table = pd.DataFrame(
    {
        "y_pred": future_m.values,
        "y_lower_95": future_m_ci.iloc[:, 0].values,
        "y_upper_95": future_m_ci.iloc[:, 1].values,
    },
    index=future_m.index,
)
future_m_table



## 6) Forecast corto plazo: viajeros diarios del proximo mes

Objetivo practico: predecir 30 dias con intervalo, comparando modelos y calidad de incertidumbre.


In [ ]:
# Pseudo-test diario: ultimos 30 dias.
train_d = df["y"].iloc[:-30]
test_d = df["y"].iloc[-30:]

# Modelo base: naive estacional semanal (lag 7).
naive_d_pred = pd.concat([train_d, test_d]).shift(7).loc[test_d.index]
naive_sigma = (train_d - train_d.shift(7)).dropna().std()
naive_d_lo = naive_d_pred - 1.96 * naive_sigma
naive_d_hi = naive_d_pred + 1.96 * naive_sigma

# Modelo SARIMAX diario con estacionalidad semanal.
sarimax_d = SARIMAX(train_d, order=(1, 0, 1), seasonal_order=(1, 0, 1, 7), enforce_stationarity=False, enforce_invertibility=False)
sarimax_d_fit = sarimax_d.fit(disp=False)
fc_d = sarimax_d_fit.get_forecast(steps=30)
sarimax_d_pred = fc_d.predicted_mean
sarimax_d_ci = fc_d.conf_int(alpha=0.05)
sarimax_d_lo = sarimax_d_ci.iloc[:, 0]
sarimax_d_hi = sarimax_d_ci.iloc[:, 1]



### 6.1) Bonus sofisticado: NHITS (deep learning especializado)

Requiere:
- `pip install neuralforecast torch`


In [ ]:
# BONUS: NHITS en diario (mismo pseudo-test de 30 dias).
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

train_nf = train_d.to_frame("y").reset_index().rename(columns={"index": "ds"})
train_nf["unique_id"] = "METRO"
train_nf = train_nf[["unique_id", "ds", "y"]]

nhits_d = NHITS(
    h=30,
    input_size=min(180, len(train_nf) - 1),
    max_steps=250,
    loss=MQLoss(level=[95]),
    random_seed=7,
)

nf_d = NeuralForecast(models=[nhits_d], freq="D")
nf_d.fit(df=train_nf)
fc_nh = nf_d.predict().reset_index().sort_values("ds")

# Nombres directos de salida NHITS en esta version.
nhits_d_pred = fc_nh["NHITS-median"].values
nhits_d_lo = fc_nh["NHITS-lo-95"].values
nhits_d_hi = fc_nh["NHITS-hi-95"].values

# Tabla comparativa diaria en pseudo-test.
summary_d = pd.DataFrame(
    {
        "Modelo": ["Naive semanal", "SARIMAX diario", "NHITS diario"],
        "MAE": [
            mean_absolute_error(test_d, naive_d_pred),
            mean_absolute_error(test_d, sarimax_d_pred),
            mean_absolute_error(test_d.values, nhits_d_pred),
        ],
        "RMSE": [
            rmse(test_d, naive_d_pred),
            rmse(test_d, sarimax_d_pred),
            rmse(test_d.values, nhits_d_pred),
        ],
        "Cobertura_IC95": [
            ((test_d >= naive_d_lo) & (test_d <= naive_d_hi)).mean(),
            ((test_d >= sarimax_d_lo) & (test_d <= sarimax_d_hi)).mean(),
            ((test_d.values >= nhits_d_lo) & (test_d.values <= nhits_d_hi)).mean(),
        ],
    }
)
summary_d["%Fuera_IC95"] = 1 - summary_d["Cobertura_IC95"]
summary_d



In [ ]:
# Visual comparativa diaria en pseudo-test (30 dias).
plt.figure(figsize=(14, 4))
plt.plot(test_d.index, test_d.values, label="Real", color="black")

plt.plot(test_d.index, sarimax_d_pred.values, label="SARIMAX", color="tab:green")
plt.fill_between(test_d.index, sarimax_d_lo.values, sarimax_d_hi.values, color="tab:green", alpha=0.2, label="IC95 SARIMAX")

plt.plot(test_d.index, nhits_d_pred, label="NHITS", color="tab:purple")
plt.fill_between(test_d.index, nhits_d_lo, nhits_d_hi, color="tab:purple", alpha=0.15, label="IC95 NHITS")

plt.title("Validacion diaria (ultimos 30 dias)")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.legend(loc="best")
plt.show()



In [ ]:
# Forecast real: proximos 30 dias con SARIMAX (practico operativo diario).
sarimax_full = SARIMAX(df["y"], order=(1, 0, 1), seasonal_order=(1, 0, 1, 7), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fc_next_30 = sarimax_full.get_forecast(steps=30)
next_30 = fc_next_30.predicted_mean
next_30_ci = fc_next_30.conf_int(alpha=0.05)

next_30_table = pd.DataFrame(
    {
        "y_pred": next_30.values,
        "y_lower_95": next_30_ci.iloc[:, 0].values,
        "y_upper_95": next_30_ci.iloc[:, 1].values,
    },
    index=next_30.index,
)
next_30_table.head(10)



## 7) Cierre conceptual

- Componente **determinista**: tendencia + estacionalidades (semanal/anual).
- Componente **aleatoria**: residuo/ruido (incertidumbre).
- Decisiones practicas: no solo punto previsto, sino rango e incertidumbre (intervalo y cobertura).
